# 💉 Antidote: Final Report Ablations (FAST VERSION)
## CS 6958 - Machine Learning Security

---

**⚡ OPTIMIZED FOR SPEED:**
- 25 epochs instead of 50
- Reuses models across experiments
- Batched similarity computation

---

⚠️ **SELECT A100 GPU:** Runtime → Change runtime type → **A100** (or L4/V100)

⏱️ **Estimated Runtime:** ~1-1.5 hours

In [ ]:
#@title 1. Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from tqdm import tqdm
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# GPU Check
assert torch.cuda.is_available(), '⚠️ No GPU! Enable: Runtime → Change runtime type → A100'
gpu_name = torch.cuda.get_device_name(0)
print(f'✅ GPU: {gpu_name}')
if 'A100' in gpu_name:
    print('🚀 A100 detected - fastest option!')
elif 'V100' in gpu_name or 'L4' in gpu_name:
    print('⚡ Fast GPU detected')
else:
    print('💡 Tip: Switch to A100 for 3x speedup')

device = torch.device('cuda')

# SPEED CONFIG
EPOCHS = 25  # Reduced from 50
N_TARGETS = 250  # Keep full for valid results
BATCH_SIZE = 256 if 'A100' in gpu_name else 128  # Larger batch for A100

In [ ]:
#@title 2. Data + Core Classes (Combined)
# Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_features = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load data
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
trainset_features = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_features)

class TruthSerumAttack:
    def __init__(self, dataset, n_targets=250, n_copies=8, seed=42):
        self.dataset = dataset
        self.n_targets = n_targets
        self.n_copies = n_copies
        np.random.seed(seed)
        self.target_indices = np.random.choice(len(dataset), n_targets, replace=False)
        
    def create_poisoned_dataset(self):
        images, labels, is_poison, original_indices = [], [], [], []
        for i in range(len(self.dataset)):
            img, label = self.dataset[i]
            images.append(img)
            labels.append(label)
            is_poison.append(False)
            original_indices.append(i)
        
        for target_idx in self.target_indices:
            img, true_label = self.dataset[target_idx]
            wrong_label = (true_label + np.random.randint(1, 10)) % 10
            for _ in range(self.n_copies):
                images.append(img.clone() if hasattr(img, 'clone') else img)
                labels.append(wrong_label)
                is_poison.append(True)
                original_indices.append(target_idx)
        return PoisonedDataset(images, labels, is_poison, original_indices)

class PoisonedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, is_poison, original_indices):
        self.images = images
        self.labels = labels
        self.is_poison = is_poison
        self.original_indices = original_indices
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]
    def get_poison_mask(self): return np.array(self.is_poison)

def create_resnet18():
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(device)

def train_model(model, trainloader, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    model.train()
    for epoch in range(epochs):
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/{epochs}')
    return model

def evaluate_accuracy(model, testloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            _, predicted = model(inputs).max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return correct / total

def compute_mi_attack(model, member_loader, nonmember_loader):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')
    member_losses, nonmember_losses = [], []
    with torch.no_grad():
        for inputs, targets in member_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            member_losses.extend(criterion(model(inputs), targets).cpu().numpy())
        for inputs, targets in nonmember_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            nonmember_losses.extend(criterion(model(inputs), targets).cpu().numpy())
    scores = np.concatenate([-np.array(member_losses), -np.array(nonmember_losses)])
    labels = np.concatenate([np.ones(len(member_losses)), np.zeros(len(nonmember_losses))])
    return roc_auc_score(labels, scores)

def evaluate_detection(predicted, ground_truth):
    p, r, f1, _ = precision_recall_fscore_support(ground_truth, predicted, average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f1, 
            'tp': int(np.sum(predicted & ground_truth)), 
            'fp': int(np.sum(predicted & ~ground_truth))}

print('✅ All classes defined')

In [ ]:
#@title 3. Antidote + Spectral Signatures Defenses
class AntidoteDefense:
    def __init__(self, similarity_threshold=0.99):
        self.similarity_threshold = similarity_threshold
        self.feature_extractor = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor.fc = nn.Identity()
        self.feature_extractor.eval().to(device)
        
    def extract_features(self, dataset):
        loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)
        features, labels = [], []
        with torch.no_grad():
            for imgs, lbls in tqdm(loader, desc='Extracting'):
                features.append(self.feature_extractor(imgs.to(device)).cpu().numpy())
                labels.append(lbls.numpy())
        return np.vstack(features), np.concatenate(labels)
    
    def detect(self, dataset_labels, features):
        n = len(features)
        norms = np.linalg.norm(features, axis=1, keepdims=True)
        features_norm = features / (norms + 1e-8)
        suspected = np.zeros(n, dtype=bool)
        
        # Process in larger batches for speed
        batch_size = 2000
        for i in tqdm(range(0, n, batch_size), desc=f'Detecting (thresh={self.similarity_threshold})'):
            end_i = min(i + batch_size, n)
            sims = features_norm[i:end_i] @ features_norm.T
            
            for j, row_idx in enumerate(range(i, end_i)):
                row_sims = sims[j].copy()
                row_sims[row_idx] = 0
                duplicates = np.where(row_sims > self.similarity_threshold)[0]
                if len(duplicates) > 0:
                    if np.any(dataset_labels[duplicates] != dataset_labels[row_idx]):
                        suspected[row_idx] = True
                    if len(duplicates) >= 3:
                        suspected[row_idx] = True
        return suspected

class SpectralSignaturesDefense:
    def __init__(self, percentile=95):
        self.percentile = percentile
        
    def detect(self, model, dataset):
        model.eval()
        activations = {}
        def hook_fn(m, i, o): activations['rep'] = o.detach()
        hook = model.avgpool.register_forward_hook(hook_fn)
        
        loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)
        reps, labels_list = [], []
        with torch.no_grad():
            for inputs, targets in tqdm(loader, desc='Spectral extraction'):
                _ = model(inputs.to(device))
                reps.append(activations['rep'].squeeze().cpu().numpy())
                labels_list.append(targets.numpy())
        hook.remove()
        
        reps = np.vstack(reps)
        labels = np.concatenate(labels_list)
        scores = np.zeros(len(reps))
        
        for c in range(10):
            mask = labels == c
            if mask.sum() < 10: continue
            centered = reps[mask] - reps[mask].mean(axis=0)
            try:
                _, _, Vt = np.linalg.svd(centered, full_matrices=False)
                scores[mask] = (centered @ Vt[0]) ** 2
            except: pass
        
        return scores > np.percentile(scores, self.percentile)

print('✅ Defenses defined')

In [ ]:
#@title 4. Create Attack Dataset (once, reuse everywhere)
print('='*60)
print('CREATING ATTACK DATASETS')
print('='*60)

# Main attack (k=8)
attack_main = TruthSerumAttack(trainset, n_targets=N_TARGETS, n_copies=8, seed=42)
poisoned_main = attack_main.create_poisoned_dataset()
gt_main = poisoned_main.get_poison_mask()

# Feature version
attack_feat = TruthSerumAttack(trainset_features, n_targets=N_TARGETS, n_copies=8, seed=42)
poisoned_feat = attack_feat.create_poisoned_dataset()

print(f'Total: {len(poisoned_main)}, Poisons: {gt_main.sum()}')

# Extract features ONCE
print('\nExtracting features (one-time)...')
defense = AntidoteDefense(0.99)
features_main, _ = defense.extract_features(poisoned_feat)
labels_main = np.array([poisoned_main[i][1] for i in range(len(poisoned_main))])
print(f'Features: {features_main.shape}')

In [ ]:
#@title 5. EXPERIMENT 1: Threshold Ablation (FAST - no retraining)
print('='*60)
print('EXP 1: Threshold Ablation')
print('='*60)

threshold_results = {}
for thresh in [0.95, 0.99, 0.999]:
    defense = AntidoteDefense(similarity_threshold=thresh)
    suspected = defense.detect(labels_main, features_main)
    metrics = evaluate_detection(suspected, gt_main)
    threshold_results[thresh] = metrics
    print(f'Thresh {thresh}: P={metrics["precision"]:.3f} R={metrics["recall"]:.3f} F1={metrics["f1"]:.3f}')

print('✅ Done (~2 min)')

In [ ]:
#@title 6. EXPERIMENT 2: Copies Ablation
print('='*60)
print('EXP 2: Number of Copies Ablation')
print('='*60)

copies_results = {}
for k in [2, 4, 8, 16]:
    print(f'\n--- k={k} copies ---')
    atk = TruthSerumAttack(trainset_features, n_targets=N_TARGETS, n_copies=k, seed=42)
    pds = atk.create_poisoned_dataset()
    gt = pds.get_poison_mask()
    
    # Extract features
    defense = AntidoteDefense(0.99)
    feats, _ = defense.extract_features(pds)
    lbls = np.array([pds[i][1] for i in range(len(pds))])
    
    suspected = defense.detect(lbls, feats)
    metrics = evaluate_detection(suspected, gt)
    copies_results[k] = metrics
    print(f'  P={metrics["precision"]:.3f} R={metrics["recall"]:.3f} F1={metrics["f1"]:.3f}')

print('\n✅ Done (~8 min)')

In [ ]:
#@title 7. EXPERIMENT 3: Train Models (for Spectral + MI eval)
print('='*60)
print('EXP 3: Training Models')
print('='*60)

testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

# Clean model
print('\n1. Training CLEAN model...')
clean_loader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
clean_model = train_model(create_resnet18(), clean_loader)
clean_acc = evaluate_accuracy(clean_model, testloader)
print(f'   Accuracy: {clean_acc*100:.1f}%')

# Poisoned model
print('\n2. Training POISONED model...')
poisoned_loader = torch.utils.data.DataLoader(poisoned_main, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
poisoned_model = train_model(create_resnet18(), poisoned_loader)
poisoned_acc = evaluate_accuracy(poisoned_model, testloader)
print(f'   Accuracy: {poisoned_acc*100:.1f}%')

# Defended model
print('\n3. Training DEFENDED model...')
best_defense = AntidoteDefense(0.99)
suspected = best_defense.detect(labels_main, features_main)
keep_mask = ~suspected
defended_ds = PoisonedDataset(
    [poisoned_main.images[i] for i in np.where(keep_mask)[0]],
    [poisoned_main.labels[i] for i in np.where(keep_mask)[0]],
    [poisoned_main.is_poison[i] for i in np.where(keep_mask)[0]],
    [poisoned_main.original_indices[i] for i in np.where(keep_mask)[0]]
)
print(f'   Removed {suspected.sum()} samples, {len(defended_ds)} remain')
defended_loader = torch.utils.data.DataLoader(defended_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
defended_model = train_model(create_resnet18(), defended_loader)
defended_acc = evaluate_accuracy(defended_model, testloader)
print(f'   Accuracy: {defended_acc*100:.1f}%')

print('\n✅ Models trained (~20 min on A100)')

In [ ]:
#@title 8. Spectral Signatures Baseline
print('='*60)
print('EXP 3b: Spectral Signatures Baseline')
print('='*60)

spectral = SpectralSignaturesDefense(percentile=95)
spectral_suspected = spectral.detect(poisoned_model, poisoned_main)
spectral_metrics = evaluate_detection(spectral_suspected, gt_main)

print(f'Spectral Signatures: P={spectral_metrics["precision"]:.3f} R={spectral_metrics["recall"]:.3f} F1={spectral_metrics["f1"]:.3f}')
print(f'Antidote (0.99):     P={threshold_results[0.99]["precision"]:.3f} R={threshold_results[0.99]["recall"]:.3f} F1={threshold_results[0.99]["f1"]:.3f}')
print('\n✅ Done (~2 min)')

In [ ]:
#@title 9. Membership Inference Evaluation
print('='*60)
print('EXP 4: Membership Inference Attack')
print('='*60)

# Target loader
target_data = [(trainset[i][0], trainset[i][1]) for i in attack_main.target_indices]
target_ds = torch.utils.data.TensorDataset(
    torch.stack([x[0] for x in target_data]),
    torch.tensor([x[1] for x in target_data])
)
target_loader = torch.utils.data.DataLoader(target_ds, batch_size=256)
nonmember_loader = testloader

clean_mi = compute_mi_attack(clean_model, target_loader, nonmember_loader)
poisoned_mi = compute_mi_attack(poisoned_model, target_loader, nonmember_loader)
defended_mi = compute_mi_attack(defended_model, target_loader, nonmember_loader)

print(f'Clean MI AUC:    {clean_mi:.3f}')
print(f'Poisoned MI AUC: {poisoned_mi:.3f}')
print(f'Defended MI AUC: {defended_mi:.3f}')

attack_gap = poisoned_mi - clean_mi
defense_reduction = poisoned_mi - defended_mi
effectiveness = (defense_reduction / attack_gap) * 100 if attack_gap > 0 else 0

print(f'\nDefense effectiveness: {effectiveness:.1f}%')

In [ ]:
#@title 10. Save Results + Generate Figures
print('='*60)
print('SAVING RESULTS')
print('='*60)

all_results = {
    'threshold_ablation': {str(k): v for k, v in threshold_results.items()},
    'copies_ablation': {str(k): v for k, v in copies_results.items()},
    'spectral_baseline': spectral_metrics,
    'end_to_end': {
        'clean_acc': clean_acc, 'poisoned_acc': poisoned_acc, 'defended_acc': defended_acc,
        'clean_mi': clean_mi, 'poisoned_mi': poisoned_mi, 'defended_mi': defended_mi,
        'effectiveness_pct': effectiveness
    }
}

with open('ablation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Generate figure
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Threshold
ax = axes[0, 0]
ts = list(threshold_results.keys())
x = np.arange(len(ts))
w = 0.25
ax.bar(x-w, [threshold_results[t]['precision'] for t in ts], w, label='Precision', color='#2a9d8f')
ax.bar(x, [threshold_results[t]['recall'] for t in ts], w, label='Recall', color='#e76f51')
ax.bar(x+w, [threshold_results[t]['f1'] for t in ts], w, label='F1', color='#264653')
ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in ts])
ax.set_xlabel('Similarity Threshold')
ax.set_ylabel('Score')
ax.set_title('(a) Threshold Ablation')
ax.legend()
ax.set_ylim(0, 1.1)

# Plot 2: Copies
ax = axes[0, 1]
cs = list(copies_results.keys())
ax.plot(cs, [copies_results[c]['f1'] for c in cs], 'o-', label='F1', color='#264653', lw=2, ms=8)
ax.plot(cs, [copies_results[c]['recall'] for c in cs], 's--', label='Recall', color='#e76f51', lw=2, ms=8)
ax.set_xlabel('Number of Copies (k)')
ax.set_ylabel('Score')
ax.set_title('(b) Copies Ablation')
ax.legend()
ax.set_ylim(0, 1.1)
ax.set_xticks(cs)

# Plot 3: Defense comparison
ax = axes[1, 0]
methods = ['Antidote', 'Spectral']
f1s = [threshold_results[0.99]['f1'], spectral_metrics['f1']]
bars = ax.bar(methods, f1s, color=['#2a9d8f', '#e9c46a'], edgecolor='black', lw=1.5)
ax.set_ylabel('Detection F1')
ax.set_title('(c) Defense Comparison')
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontweight='bold')

# Plot 4: MI AUC
ax = axes[1, 1]
models = ['Clean', 'Poisoned', 'Defended']
aucs = [clean_mi, poisoned_mi, defended_mi]
bars = ax.bar(models, aucs, color=['#2a9d8f', '#e63946', '#f4a261'], edgecolor='black', lw=1.5)
ax.axhline(0.5, color='gray', ls='--', label='Random')
ax.set_ylabel('MI Attack AUC')
ax.set_title('(d) Membership Inference')
ax.set_ylim(0, 1.0)
ax.legend()
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('ablation_figures.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Saved: ablation_results.json, ablation_figures.png')

In [ ]:
#@title 11. Summary Tables (copy to report)
print('='*70)
print('TABLES FOR FINAL REPORT')
print('='*70)

print('\n### Table 1: Threshold Ablation')
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-'*48)
for t in threshold_results:
    r = threshold_results[t]
    print(f'{t:<12} {r["precision"]:<12.3f} {r["recall"]:<12.3f} {r["f1"]:<12.3f}')

print('\n### Table 2: Copies Ablation')
print(f'{"k":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-'*48)
for c in copies_results:
    r = copies_results[c]
    print(f'{c:<12} {r["precision"]:<12.3f} {r["recall"]:<12.3f} {r["f1"]:<12.3f}')

print('\n### Table 3: Defense Comparison')
print(f'{"Method":<20} {"F1":<12}')
print('-'*32)
print(f'{"Antidote (Ours)":<20} {threshold_results[0.99]["f1"]:<12.3f}')
print(f'{"Spectral Signatures":<20} {spectral_metrics["f1"]:<12.3f}')

print('\n### Table 4: End-to-End')
print(f'{"Model":<15} {"Accuracy":<12} {"MI AUC":<12}')
print('-'*39)
print(f'{"Clean":<15} {clean_acc*100:<12.1f} {clean_mi:<12.3f}')
print(f'{"Poisoned":<15} {poisoned_acc*100:<12.1f} {poisoned_mi:<12.3f}')
print(f'{"Defended":<15} {defended_acc*100:<12.1f} {defended_mi:<12.3f}')
print(f'\nDefense effectiveness: {effectiveness:.1f}%')

In [ ]:
#@title 12. Download
from google.colab import files
files.download('ablation_results.json')
files.download('ablation_figures.png')
print('✅ Done!')